> **Jupyter 학습 안내**
>
> 이 Notebook은 Course의 상세 학습노트입니다. 본문과 수식을 먼저 읽고 Python 예제 셀을 한 단계씩 실행하세요.
> 코드 셀은 개념을 보여주는 작은 예제로, 필요한 데이터와 변수는 바로 앞 설명을 확인해야 합니다.
> 처음부터 끝까지 실행하는 통합 실습은 저장소의 notebooks와 notebooks/data_analysis를 사용합니다.
> 예제 결과를 예상한 뒤 실행하고, 값·조건·열 이름을 바꾸어 결과 차이를 기록하세요.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "courses").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 안에서 Notebook을 실행하세요.")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("학습 저장소:", ROOT)

# Course 05 학습노트: 탐색적 데이터 분석과 통계적 추론

## 1. EDA의 목적

탐색적 데이터 분석(EDA)은 예쁜 그래프를 만드는 단계가 아니라 데이터 구조, 품질, 분포, 관계, 예외를 이해하고 다음 분석 질문을 구체화하는 과정이다. 권장 학습시간은 이론 100분, 예제 60분, Lab 03·04 180분이다.

EDA에서는 다음 순서를 권장한다.

1. 관측 단위와 데이터 범위를 확인한다.
2. 단일 변수의 분포를 확인한다.
3. 집단 간 차이를 확인한다.
4. 두 변수의 관계를 확인한다.
5. 결과의 불확실성과 대안 설명을 검토한다.

## 2. 분석 질문을 측정 가능한 문장으로 바꾸기

“좋은 상권은 어디인가?”보다 다음 질문이 분석 가능하다.

- 행정동별 유동인구 중앙값은 얼마나 다른가?
- 카페 수와 유동인구 사이에 선형 관계가 있는가?
- 20대 인구 대비 카페 수가 높은 지역은 어디인가?
- 추천 순위는 가중치가 10% 바뀌어도 유지되는가?

질문에는 대상, 변수, 비교 기준, 시간·공간 범위가 있어야 한다.

## 3. 분포 요약

평균은 이상값에 민감하고 중앙값은 강건하다. 표준편차는 평균 주변의 퍼짐을, IQR은 가운데 50%의 범위를 나타낸다.

In [ ]:
stats = df["유동인구"].agg(["count", "mean", "median", "std", "min", "max"])
q1, q3 = df["유동인구"].quantile([.25, .75])
stats["iqr"] = q3 - q1
print(stats)

히스토그램은 구간별 빈도, 상자그림은 중앙값·사분위·이상 후보를 보여준다. 두 그래프는 서로 대체하지 않는다.

## 4. 시각화 선택

| 질문 | 권장 그래프 | 주의 |
|---|---|---|
| 범주 간 크기 비교 | 정렬 막대 | 축과 단위 표시 |
| 시간 변화 | 선 | 시간 간격 확인 |
| 두 수치 관계 | 산점도 | 겹침·이상값 확인 |
| 분포 비교 | 상자·바이올린 | 표본 수 함께 표시 |
| 공간 패턴 | 지도 | 면적·인구 차이 고려 |

In [ ]:
import plotly.express as px

fig = px.scatter(
    df, x="카페수", y="유동인구", size="총인구",
    hover_name="행정동명", trendline="ols",
    labels={"카페수": "카페 수(개)", "유동인구": "유동인구 지수"},
    title="행정동별 카페 수와 유동인구",
)
fig.show()

## 5. 표본과 표준오차

표본평균은 표본에 따라 달라진다. 독립 표본의 크기가 커지면 평균의 표준오차가 대략 다음과 같이 감소한다.

$$SE(\bar{x})=\frac{s}{\sqrt{n}}$$

표본 크기를 4배로 늘려야 표준오차가 절반이 된다. 데이터 행 수가 많아도 편향된 표본이면 대표성은 좋아지지 않는다.

## 6. 부트스트랩 신뢰구간

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
x = df["월매출"].dropna().to_numpy()
boot_means = [rng.choice(x, size=len(x), replace=True).mean() for _ in range(2000)]
ci = np.percentile(boot_means, [2.5, 97.5])
print("95% bootstrap CI:", ci)

신뢰구간은 표본추출 절차의 불확실성을 표현한다. 원자료가 모집단을 대표하지 못하는 편향은 부트스트랩으로 해결되지 않는다.

## 7. 가설검정과 효과크기

두 집단 평균을 비교할 때 귀무가설은 보통 `두 모집단 평균이 같다`이다. p값은 귀무가설 아래 현재와 같거나 더 극단적인 통계량이 나올 확률이다.

In [ ]:
from scipy import stats

a = df.loc[df["업종"] == "카페", "월매출"].dropna()
b = df.loc[df["업종"] == "음식점", "월매출"].dropna()
t, p = stats.ttest_ind(a, b, equal_var=False)

p값이 작아도 차이가 실무적으로 작을 수 있으므로 Cohen's d와 원 단위 평균 차이를 함께 보고한다.

$$d=\frac{\bar{x}_1-\bar{x}_2}{s_{pooled}}$$

## 8. 상관관계

Pearson 상관계수는 선형 관계를 측정한다.

$$r=\frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum(x_i-\bar{x})^2\sum(y_i-\bar{y})^2}}$$

상관계수는 이상값, 비선형 관계, 집단 혼합의 영향을 받는다. 전체 상관과 업종별 상관이 다를 수 있으므로 산점도를 먼저 확인한다.

## 9. 그래프 해석의 세 문장

모든 결과는 다음을 구분해 쓴다.

- 관찰: “신흥동 유동인구는 24,800으로 비교 지역 중 가장 높다.”
- 해석: “상업 활동이 활발할 가능성을 시사한다.”
- 한계: “시간대별 보행량이 아니라 합성 지표이므로 실제 방문객 수로 단정할 수 없다.”

## 10. 단계별 실습

1. 수치형 변수의 분포표와 결측률을 만든다.
2. 분석 질문 5개를 분포·비교·관계·공간·불확실성으로 나눈다.
3. 각 질문에 맞는 그래프를 만들고 단위·기준을 표시한다.
4. Lab 03에서 표본 크기별 표준오차를 비교한다.
5. Lab 04에서 p값, 효과크기, 상관계수를 계산한다.
6. 각 결과에 관찰·해석·한계를 작성한다.

## 학습 마무리

1. 이 Course의 핵심 개념 세 가지를 본인의 말로 정리한다.
2. 수식 하나를 작은 숫자로 손계산하고 Python 결과와 비교한다.
3. 예제 코드의 입력이나 조건을 하나 바꾸어 결과 차이를 설명한다.
4. quiz.md에 먼저 답한 뒤 quiz_answer.md와 비교한다.
5. assignment.md와 연결된 실습 Notebook을 Restart & Run All로 확인한다.